# Tutorial 2: PDE Basics - Pattern Formation with Differential Equations

**Learning Objectives:**
- Understand how PDEs create patterns
- Run classical pattern-forming PDEs (Gray-Scott, Chhabra)
- Use the PDE solver infrastructure
- Explore reaction-diffusion dynamics
- Visualize PDE solutions

**Time:** 1 hour

**Prerequisites:** Tutorial 1, basic understanding of derivatives

---

## What are Pattern-Forming PDEs?

Partial Differential Equations (PDEs) describe how quantities change in space and time:

$$\frac{\partial u}{\partial t} = \text{reaction}(u) + \text{diffusion}(u) + \text{advection}(u)$$

**Key Components:**
- **Reaction:** Local interactions (e.g., chemical reactions)
- **Diffusion:** Spreading in space (∇²u)
- **Advection:** Transport/flow (∇·(uv))

Classic examples:
- **Gray-Scott:** Spots and stripes
- **Turing patterns:** Animal coat patterns
- **Navier-Stokes:** Fluid flow
- **Keller-Segel:** Chemotaxis and cell aggregation

These PDEs create emergent patterns from simple rules!

## Setup

In [ ]:
import sys
sys.path.append('..')

import jax
import jax.numpy as jnp
import jax.random as jr
from einops import rearrange
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import time

# PDE-specific imports
from PDE.model.solver.semidiscrete_solver import PDE_solver
from PDE.model.fixed_models.update_gray_scott import F as F_gray_scott
from PDE.model.fixed_models.update_chhabra import F as F_chhabra
from Common.model.spatial_operators import Ops
from Common.utils import my_animate

print(f"JAX version: {jax.__version__}")

## Part 1: Understanding the Gray-Scott Model

The Gray-Scott model is a classic reaction-diffusion system:

$$\frac{\partial U}{\partial t} = D_U \nabla^2 U - UV^2 + \alpha(1-U)$$

$$\frac{\partial V}{\partial t} = D_V \nabla^2 V + UV^2 - (\alpha + \gamma)V$$

Where:
- U and V are two chemical species
- D_U, D_V are diffusion coefficients
- α (alpha) is the feed rate
- γ (gamma) is the kill rate

Different parameter values produce spots, stripes, or chaotic patterns!

### Creating Initial Conditions

In [ ]:
# Grid configuration
SIZE = 128
N_SPECIES = 2  # U and V

# Create random initial conditions
key = jr.PRNGKey(int(time.time()))
x0 = jr.uniform(key, shape=(N_SPECIES, SIZE, SIZE))

# Smooth the initial conditions using averaging
op = Ops(PADDING="CIRCULAR", dx=1.0, KERNEL_SCALE=3)
for i in range(5):
    x0 = op.Average(x0)

# Set up the initial state:
# U (species 0): mostly 1 where V is low
# V (species 1): mostly 0
x0 = x0.at[0].set(jnp.where(x0[0] > 0.51, 0.0, 1.0))
x0 = op.Average(x0)
x0 = x0.at[1].set(1 - x0[0])

print(f"Initial state shape: {x0.shape}")

# Visualize initial conditions
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(x0[0], cmap='viridis')
axes[0].set_title('Species U (initial)')
axes[0].axis('off')

axes[1].imshow(x0[1], cmap='plasma')
axes[1].set_title('Species V (initial)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

### Running the Gray-Scott Model

In [ ]:
# Define the Gray-Scott PDE
func = F_gray_scott(
    PADDING="CIRCULAR",  # Periodic boundaries
    dx=0.5,              # Spatial resolution
    alpha=0.0623,        # Feed rate (try values 0.01-0.08)
    gamma=0.06268        # Kill rate (try values 0.05-0.07)
)

# Create PDE solver
solver = PDE_solver(
    func,
    dt=0.2,           # Time step
    SOLVER="heun",    # Numerical method (options: "euler", "heun", "dopri5")
    ADAPTIVE=False    # Use adaptive stepping?
)

# Solve the PDE
T, Y = solver(
    ts=jnp.linspace(0, 10000, 200),  # Time points
    y0=x0                             # Initial condition
)

print(f"Time points shape: {T.shape}")
print(f"Solution shape: {Y.shape}")  # (Time, Species, Height, Width)

### Visualizing Results

In [ ]:
# Show evolution over time (every 20th frame)
frames_to_show = Y[::20, 0]  # Show species U

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < len(frames_to_show):
        ax.imshow(frames_to_show[i], cmap='viridis')
        ax.set_title(f't = {T[i*20]:.1f}')
    ax.axis('off')

plt.suptitle('Gray-Scott Model Evolution (Species U)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Create animation (this uses the helper function from Common.utils)
print("Creating animation...")
my_animate(Y[:, :1], clip=False)  # Animate species U

## Part 2: Exploring Different Parameter Regimes

Different parameters produce different patterns. Let's explore!

In [ ]:
# Dictionary of interesting parameter sets
parameter_sets = {
    "Spots": {"alpha": 0.062, "gamma": 0.061},
    "Stripes": {"alpha": 0.046, "gamma": 0.063},
    "Waves": {"alpha": 0.014, "gamma": 0.054},
    "Chaos": {"alpha": 0.026, "gamma": 0.051},
}

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()

for idx, (name, params) in enumerate(parameter_sets.items()):
    print(f"Running {name}...")
    
    # Create solver with these parameters
    func = F_gray_scott(
        PADDING="CIRCULAR",
        dx=0.5,
        alpha=params["alpha"],
        gamma=params["gamma"]
    )
    solver = PDE_solver(func, dt=0.2)
    
    # Solve
    T, Y = solver(ts=jnp.linspace(0, 10000, 100), y0=x0)
    
    # Plot final state
    axes[idx].imshow(Y[-1, 0], cmap='viridis')
    axes[idx].set_title(f'{name}\nα={params["alpha"]}, γ={params["gamma"]}')
    axes[idx].axis('off')

plt.suptitle('Gray-Scott Model: Different Parameter Regimes', fontsize=16)
plt.tight_layout()
plt.show()

## Part 3: The Chhabra Model

The Chhabra model is another pattern-forming PDE that produces interesting dynamics:

In [ ]:
# Create new initial conditions for Chhabra
key = jr.PRNGKey(42)
x0_chhabra = jr.uniform(key, shape=(2, SIZE, SIZE))

# Smooth
for i in range(5):
    x0_chhabra = op.Average(x0_chhabra)

# Define Chhabra PDE
func_chhabra = F_chhabra(
    PADDING="CIRCULAR",
    dx=0.5,
    D1=1.0,    # Diffusion coefficient for species 1
    D2=10.0,   # Diffusion coefficient for species 2
    k1=0.5,    # Reaction rate parameters
    k2=1.0,
    k3=1.0
)

solver_chhabra = PDE_solver(func_chhabra, dt=0.1)

print("Running Chhabra model...")
T_chhabra, Y_chhabra = solver_chhabra(
    ts=jnp.linspace(0, 100, 200),
    y0=x0_chhabra
)

print(f"Chhabra solution shape: {Y_chhabra.shape}")

In [ ]:
# Visualize Chhabra evolution
frames_to_show = Y_chhabra[::20]

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, ax in enumerate(axes):
    if i < len(frames_to_show):
        # Show both species as RGB (U=red, V=green)
        rgb = jnp.stack([
            frames_to_show[i, 0],
            frames_to_show[i, 1],
            jnp.zeros_like(frames_to_show[i, 0])
        ], axis=-1)
        ax.imshow(rgb)
        ax.set_title(f't = {T_chhabra[i*20]:.1f}')
    ax.axis('off')

plt.suptitle('Chhabra Model Evolution (U=red, V=green)', fontsize=16)
plt.tight_layout()
plt.show()

## Part 4: Understanding the PDE Solver

Let's understand how the solver works under the hood.

### Spatial Operators

The `Ops` class provides spatial derivatives using finite differences:

In [ ]:
# Create a test field with a Gaussian bump
x = jnp.linspace(-5, 5, 64)
y = jnp.linspace(-5, 5, 64)
X, Y = jnp.meshgrid(x, y)
field = jnp.exp(-(X**2 + Y**2) / 2.0)
field = field[None, ...]  # Add channel dimension

# Create operator
op = Ops(PADDING="ZEROS", dx=1.0)

# Compute derivatives
gradient = op.Grad(field)
laplacian = op.Lap(field)
averaged = op.Average(field)

# Visualize
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(field[0], cmap='viridis')
axes[0, 0].set_title('Original Field')
axes[0, 0].axis('off')

axes[0, 1].imshow(gradient[0, 0], cmap='RdBu')  # x-gradient
axes[0, 1].set_title('∂/∂x (Gradient X)')
axes[0, 1].axis('off')

axes[0, 2].imshow(gradient[0, 1], cmap='RdBu')  # y-gradient
axes[0, 2].set_title('∂/∂y (Gradient Y)')
axes[0, 2].axis('off')

axes[1, 0].imshow(laplacian[0], cmap='RdBu')
axes[1, 0].set_title('∇² (Laplacian)')
axes[1, 0].axis('off')

axes[1, 1].imshow(averaged[0], cmap='viridis')
axes[1, 1].set_title('Average (Smoothed)')
axes[1, 1].axis('off')

# Magnitude of gradient
grad_mag = jnp.sqrt(gradient[0, 0]**2 + gradient[0, 1]**2)
axes[1, 2].imshow(grad_mag, cmap='viridis')
axes[1, 2].set_title('|∇| (Gradient Magnitude)')
axes[1, 2].axis('off')

plt.suptitle('Spatial Operators on a Gaussian Field', fontsize=16)
plt.tight_layout()
plt.show()

### Solver Methods

The `PDE_solver` supports different numerical integration methods:

In [ ]:
# Compare different solvers
solvers = ["euler", "heun", "dopri5"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

func = F_gray_scott(PADDING="CIRCULAR", dx=0.5, alpha=0.062, gamma=0.061)

for idx, solver_name in enumerate(solvers):
    print(f"Running {solver_name} solver...")
    
    solver = PDE_solver(func, dt=0.5, SOLVER=solver_name, ADAPTIVE=False)
    T, Y = solver(ts=jnp.linspace(0, 5000, 100), y0=x0)
    
    axes[idx].imshow(Y[-1, 0], cmap='viridis')
    axes[idx].set_title(f'{solver_name.upper()} Solver')
    axes[idx].axis('off')

plt.suptitle('Comparison of Different Numerical Solvers', fontsize=16)
plt.tight_layout()
plt.show()

print("\nSolver descriptions:")
print("- Euler: Simple, fast, less accurate")
print("- Heun: Second-order, good balance")
print("- Dopri5: Adaptive Runge-Kutta, most accurate")

## Part 5: Creating Custom PDEs

You can create your own PDE by defining the right-hand side (RHS) function:

In [ ]:
import equinox as eqx

class SimpleDiffusion(eqx.Module):
    """Simple diffusion equation: ∂u/∂t = D∇²u"""
    
    D: float  # Diffusion coefficient
    op: Ops
    
    def __init__(self, D=1.0, PADDING="CIRCULAR", dx=1.0):
        self.D = D
        self.op = Ops(PADDING=PADDING, dx=dx)
    
    def __call__(self, t, y, args):
        """Compute du/dt = D∇²u"""
        laplacian = self.op.Lap(y)
        return self.D * laplacian


# Test our custom PDE
custom_pde = SimpleDiffusion(D=0.5, PADDING="CIRCULAR", dx=0.5)
solver_custom = PDE_solver(custom_pde, dt=0.1)

# Initial condition: random noise
key = jr.PRNGKey(123)
x0_custom = jr.normal(key, shape=(1, SIZE, SIZE))

print("Running custom diffusion PDE...")
T_custom, Y_custom = solver_custom(
    ts=jnp.linspace(0, 50, 100),
    y0=x0_custom
)

# Show diffusion smoothing noise over time
frames_to_show = [0, 25, 50, 75, 99]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i, frame_idx in enumerate(frames_to_show):
    axes[i].imshow(Y_custom[frame_idx, 0], cmap='RdBu', vmin=-2, vmax=2)
    axes[i].set_title(f't = {T_custom[frame_idx]:.1f}')
    axes[i].axis('off')

plt.suptitle('Pure Diffusion: Smoothing Random Noise', fontsize=16)
plt.tight_layout()
plt.show()

## Part 6: Exercises

### Exercise 1: Parameter Exploration
Explore the Gray-Scott parameter space:
- Try α ∈ [0.01, 0.08] and γ ∈ [0.05, 0.07]
- Create a grid of results
- Find the boundary between spots and stripes

### Exercise 2: Different Initial Conditions
Try different initial conditions:
- Single localized spot
- Periodic stripes
- Random noise of different frequencies

### Exercise 3: Spatial Resolution
How does changing `dx` (spatial resolution) affect:
- Pattern wavelength?
- Numerical stability?
- Computation time?

### Exercise 4: Reaction-Diffusion Advection
Create a PDE with advection:
```python
∂u/∂t = D∇²u + R(u) - v·∇u
```
where v is a velocity field.

### Exercise 5: Multi-Species Systems
Extend to 3+ species and explore competitive/cooperative dynamics.

## Key Takeaways

1. **PDEs create patterns through local interactions:**
   - Reaction (local chemistry)
   - Diffusion (spatial spreading)
   - Advection (transport)

2. **The `PDE_solver` class:**
   - Wraps diffrax ODE solvers
   - Handles spatial discretization
   - Supports multiple numerical methods

3. **Spatial operators (`Ops`):**
   - Implement finite difference approximations
   - Support different boundary conditions
   - Enable building complex PDEs

4. **Parameter sensitivity:**
   - Small parameter changes → different patterns
   - Exploration reveals pattern "phase space"

## Next Steps

- **Tutorial 3:** Advanced NCA architectures
- **Tutorial 4:** Neural PDEs - learning PDE parameters from data
- **Demo:** `train_nca_to_pde_demo.py` - training NCA to mimic PDEs

## Additional Resources

- [Reaction-Diffusion Systems (Scholarpedia)](http://www.scholarpedia.org/article/Reaction-diffusion_systems)
- [Karl Sims' Reaction-Diffusion Tutorial](https://www.karlsims.com/rd.html)
- [Diffrax Documentation](https://docs.kidger.site/diffrax/)

---

**Congratulations!** You now understand how to simulate pattern-forming PDEs!